# 7.1 看懂 Agent 整体架构

## 概述

前面的课程已经讲过怎样把一个公式写成算子、检查计算结果，以及测量和优化性能。掌握这些知识后，我们也可以把一项完整的算子开发任务交给 Agent。任务的起点可能只是一句 prompt：

> 这是我的 XX 算子需求，请帮我实现它。

### Quick Start：直接运行完整流程

如果你现在就想把工作流用起来，可以直接跳到 [7.3 的课后练习](07.03_practice.ipynb)，参考[完整流程练习](src/exercise_task.md)运行。练习已经准备好需求文档、指定案例、启动提示词，以及关键暂停节点的推荐回复。替换任务目录后，即可让 Agent 开始完成算子交付。

### 为什么还要学习本章？

Agent 启动后，会继续完成需求分析、方案设计、代码生成、系统测试、性能调优和代码检查。模型、软件版本和设备状态不同，实际过程和结果也可能不同。学习本章，可以帮助我们：

- 看懂 Agent 当前处于哪个阶段，以及该阶段应该交付什么；
- 判断测试、性能和 Review 证据能否支持当前结论；
- 在暂停、失败或结果不达标时，知道应该补充信息、继续推进还是返回修正；
- 核对最终代码与验证证据是否对应，判断算子是否满足交付要求。

本章围绕这些能力，介绍 Agent 完成算子交付的完整过程。本节先沿着一张架构图认识整体流程；第二节学习各阶段怎样推进，并重点了解设计、ST 和调优阶段的核心方法；第三节以第三章的行 Softmax 算子为例，跟随它从设计走向最终交付，练习怎样查看阶段产物、判断结果并确定下一步行动。章节最后，大家将运用本章所学，亲自运行一次完整流程。

*我们学习的工作流为：**基于 Skills 编排的面向 TileLang 算子的全生命周期交付 Agent 工作流**。后文简称“算子交付工作流”。*

### 学习目标

- 掌握算子交付工作流整体架构
- 理清各阶段的作用
- 了解 Skills 在工作流中的作用


[本章导航](README.md) · [下一节](07.02_agent_workflow.ipynb)

本节约 **15 分钟**，建议在学完前六章后学习。本章学习路线是：本节看懂架构 → 7.2 学习各阶段原理 → 7.3 学习真实算子的 Agent 交付过程。

## 1. 了解工作流

![Skill 架构](images/07.01_skill_architecture_simple.png)

如图所示，算子交付工作流包含五个实际工作阶段：“设计 → 生成 → ST → 调优 → 代码 Review”，全部完成后形成最终交付。设计阶段给出实现方案，生成阶段交付代码和指定测试，ST 补齐适用的验证，调优生成并筛选优化候选，代码 Review 检查最终交付代码的格式和 CI 规范。阶段之间也有反馈与修正机制：ST 发现实现问题时，返回生成阶段修改；最终交付所需的性能证据未达标时，返回调优阶段继续分析。

核心概念：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">名称</th>
<th style="text-align: left; vertical-align: top;">在这套工程里做什么</th>
<th style="text-align: left; vertical-align: top;">一个具体例子</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">Agent</td>
<td style="text-align: left; vertical-align: top;">理解任务、推进工作</td>
<td style="text-align: left; vertical-align: top;">读完需求后，先准备设计，再开始实现</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">Skill</td>
<td style="text-align: left; vertical-align: top;">提供某类工作的专业指导</td>
<td style="text-align: left; vertical-align: top;"><code>tilelang-op-test-design</code> 指导测试设计与覆盖检查</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">工具</td>
<td style="text-align: left; vertical-align: top;">实际执行读文件、改代码、编译和测试等操作</td>
<td style="text-align: left; vertical-align: top;">用命令运行 pytest，得到真实测试结果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">工作流</td>
<td style="text-align: left; vertical-align: top;">规定这些工作如何衔接、何时继续、何时返回处理</td>
<td style="text-align: left; vertical-align: top;">ST 发现实现错误后，修复代码并重新验证</td>
</tr>
</tbody>
</table>


## 2. 工作流中各阶段职责

这一节只回答三个问题：每个阶段接收什么、负责什么、向后交付什么。具体怎样完成这些工作，留到 7.2 展开。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">阶段</th>
<th style="text-align: left; vertical-align: top;">核心输入</th>
<th style="text-align: left; vertical-align: top;">主要职责</th>
<th style="text-align: left; vertical-align: top;">核心输出</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><strong>设计</strong></td>
<td style="text-align: left; vertical-align: top;">算子需求信息及可用依据</td>
<td style="text-align: left; vertical-align: top;">明确算子行为，形成实现与验证方案</td>
<td style="text-align: left; vertical-align: top;"><code>design.md</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><strong>生成</strong></td>
<td style="text-align: left; vertical-align: top;"><code>design.md</code></td>
<td style="text-align: left; vertical-align: top;">将设计落实为可运行实现，并通过指定案例</td>
<td style="text-align: left; vertical-align: top;">算子代码、指定 pytest 及结果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><strong>ST</strong></td>
<td style="text-align: left; vertical-align: top;">已确认的算子要求、<code>design.md</code>、当前代码和已有测试</td>
<td style="text-align: left; vertical-align: top;">审查测试可信度，补齐适用验证</td>
<td style="text-align: left; vertical-align: top;">补充测试、ST 报告与执行证据</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><strong>调优</strong></td>
<td style="text-align: left; vertical-align: top;">通过 ST 的实现、性能案例和目标</td>
<td style="text-align: left; vertical-align: top;">根据实测数据形成并筛选优化候选</td>
<td style="text-align: left; vertical-align: top;">最终代码、性能报告</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><strong>代码 Review</strong></td>
<td style="text-align: left; vertical-align: top;">最终代码、测试文件和仓库规范</td>
<td style="text-align: left; vertical-align: top;">检查格式、lint 与 CI 规范</td>
<td style="text-align: left; vertical-align: top;">格式检查报告、最终交付文件</td>
</tr>
</tbody>
</table>

这些阶段不是彼此孤立的。前一阶段形成的核心文件，会成为后一阶段继续工作的依据，具体交接关系如下：

![核心文件交接](images/07.01_handoff_horizontal.png)

*图中只保留各阶段向后传递的核心文件。实际工作还会记录设备、后端、软件版本和执行命令等信息。同一份文件可能被多个阶段使用；文件发生变化后，后续结论也要用新版本重新验证。最终交付是五个阶段完成后的结果，并不是一个独立的 Skill 阶段。*

## 3. Skills 简介

Skills 的名称可以帮助我们定位资料，不需要逐个背下来。下面使用 CANNBot 中的正式名称，按工作问题认识它们。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">当前要解决的问题</th>
<th style="text-align: left; vertical-align: top;">主要支撑</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">怎样把算子需求变成具体方案？</td>
<td style="text-align: left; vertical-align: top;"><code>tilelang-op-design</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">怎样根据设计生成实现和基础测试？</td>
<td style="text-align: left; vertical-align: top;"><code>tilelang-op-develop</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">怎样设计测试、精度标准和覆盖范围？</td>
<td style="text-align: left; vertical-align: top;"><code>tilelang-op-test-design</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">怎样采集 TileLang 算子的性能数据？</td>
<td style="text-align: left; vertical-align: top;"><code>tilelang-op-profiling</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">怎样分析瓶颈并形成优化方案？</td>
<td style="text-align: left; vertical-align: top;"><code>tilelang-perf-optimization</code>、<code>tilelang-performance-best-practices</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">怎样查询 TileLang API 和选择编程模式？</td>
<td style="text-align: left; vertical-align: top;"><code>tilelang-api-best-practices</code>、<code>tilelang-programming-model-guide</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">怎样检查最终代码？</td>
<td style="text-align: left; vertical-align: top;"><code>tilelang-review</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">怎样检查环境和确认硬件能力？</td>
<td style="text-align: left; vertical-align: top;"><code>tilelang-env-check</code>、<code>npu-arch</code></td>
</tr>
</tbody>
</table>



## 小结

- 五个阶段分别负责形成设计、生成实现、系统测试、改进性能和检查代码规范，共同支撑算子的最终交付。
- 阶段之间既传递产物，也传递要求；出现问题后，需要返回相应位置修正并重新验证。
- 工作流根据阶段目标调用相应 Skill；一个阶段可以使用多个 Skill，同一个 Skill 也可以支撑多个阶段。

## 课后练习

1. 将 `design.md`、算子代码、补充 pytest、逐案例性能表和代码格式检查报告对应到相关阶段。哪些产物会被多个阶段使用？
2. 某个优化候选更快，但数值测试失败。能否直接交付？指出它可能返回的位置。

运行下面的代码单元即可查看练习解析。


In [ ]:
from pathlib import Path
print(Path("answer/07.01_answer.md").read_text(encoding="utf-8"))


[返回章节目录](README.md) · [下一节：Skills 编排原理](07.02_agent_workflow.ipynb)